# Faruq-v3 — Hard-Pair Identifiability Audit

Post-training visual/annotation diagnostic only. **No training, no test, no new inference.** The notebook reuses the existing CPE0/CIR0 validation object-events and the 17 frozen undirected hard-confusion families from the pre-Circle consensus audit.

For each hard pair it generates contact sheets for: `shared_pair_error`, `CPE0-only pair error`, `CIR0-only pair error`, and `both correct` reference examples. These images support human review of visual overlap / annotation identifiability, but the script does **not** declare any label wrong automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection')
BRANCH='agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    r=subprocess.run(clone)
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
BASE_REQUIRED=('bundles/faruq-development-v3-grouped.tar',)
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=BASE_REQUIRED)
ARCHIVE=require_project_artifact(PROJECT_ROOT,BASE_REQUIRED[0])
EXPERIMENTS=PROJECT_ROOT/'experiments'
def find_unique(filename, parent_hint):
    matches=[p for p in EXPERIMENTS.rglob(filename) if parent_hint in p.parts]
    if len(matches)!=1: raise RuntimeError(f'{filename}: expected 1 under {parent_hint}, found {len(matches)}: {matches}')
    return matches[0]
CPE0_EVENT=find_unique('CPE0_seed42_events.json','faruq-v3-circle-cpe-hard-confusion-reduction-v1')
CIR0_EVENT=find_unique('CIR0_seed42_events.json','faruq-v3-circle-cpe-hard-confusion-reduction-v1')
CONSENSUS_JSON=find_unique('cross_model_hard_confusion_consensus.json','faruq-v3-cross-model-hard-confusion-consensus-seed42-v1')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA_ROOT/'faruq_grouped_summary.json').is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT=EXPERIMENTS/'faruq-v3-hard-pair-identifiability-audit-v1'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
SUMMARY=OUTPUT_ROOT/'hard_pair_identifiability_audit.json'
print('CPE0 EVENT:',CPE0_EVENT)
print('CIR0 EVENT:',CIR0_EVENT)
print('CONSENSUS:',CONSENSUS_JSON)
print('OUTPUT:',OUTPUT_ROOT)


In [ ]:
command=[sys.executable,'-m','pytest','-q','tests/test_hard_pair_identifiability_audit.py']
print('STATIC CHECK:',' '.join(command))
subprocess.run(command,cwd=REPO,check=True)


In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.analysis.hard_pair_identifiability_audit',
         '--cpe0-event',str(CPE0_EVENT),'--cir0-event',str(CIR0_EVENT),
         '--consensus-json',str(CONSENSUS_JSON),'--data-root',str(DATA_ROOT),
         '--output-root',str(OUTPUT_ROOT)]
subprocess.run(command,cwd=REPO,check=True)
result=json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split']=='val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
assert result['n_frozen_undirected_hard_pairs']==17
assert result['screening_decision_remains']=='STOP_CIRCLE_CPE'


In [ ]:
import pandas as pd
from IPython.display import display
rows=[]
for r in result['families']:
    rows.append({'family':r['family'],'consensus_support':r['consensus_total_support_seed42_six_variants'],
                 'pair_gt':r['gt_instances_in_pair'],'CPE0_pair_errors':r['cpe0_pair_errors'],
                 'CIR0_pair_errors':r['cir0_pair_errors'],'delta_CIR0_minus_CPE0':r['delta_cir0_minus_cpe0'],
                 'shared':r['categories'].get('shared_pair_error',0),
                 'CPE0_only':r['categories'].get('cpe0_pair_only',0),
                 'CIR0_only':r['categories'].get('cir0_pair_only',0),
                 'both_correct':r['categories'].get('both_correct',0)})
display(pd.DataFrame(rows))
print('CATEGORY TOTALS:',result['category_totals_across_pairs'])
print('GUARDRAIL:',result['interpretation_guardrail'])
print('SCREENING DECISION REMAINS:',result['screening_decision_remains'])


In [ ]:
from IPython.display import Image, Markdown, display
shown=0
for row in result['families']:
    if shown>=6: break
    sheets=row.get('contact_sheets',{})
    if not sheets: continue
    display(Markdown('### '+row['family']))
    for category in ('shared_pair_error','cir0_pair_only','cpe0_pair_only','both_correct'):
        p=sheets.get(category)
        if p and Path(p).is_file():
            display(Markdown('**'+category+'**'))
            display(Image(filename=p,width=900))
    shown+=1
ZIP=shutil.make_archive(str(OUTPUT_ROOT/'contact_sheets'),'zip',root_dir=OUTPUT_ROOT/'contact_sheets')
print('SUMMARY:',SUMMARY)
print('CONTACT SHEETS ZIP:',ZIP)
print('Tidak ada training dan test tidak dibuka.')
